# Assignment 1 · The evaluation engine
**Quantitative Asset Management · Frankfurt School of Finance & Management · 2026**

| Name | Student number |
|---|---|
| _Name 1_ | _xxxxxxx_ |
| _Name 2_ | _xxxxxxx_ |
| _Name 3_ | _xxxxxxx_ |
| _Name 4_ | _xxxxxxx_ |

**AI coding assistants used:** OpenAI ChatGPT (Codex) and Anthropic Claude (Claude Code), for code, checks and drafting of the written answers. All numbers and answers were checked by the group.

**How to run:** keep this notebook in the folder that contains `QAM26_shrinkage.py`, `QAM26_optimization.py` and the folder `QAM26_data/` (the layout of `QAM26_data.zip`; CSVs placed directly beside the notebook also work). Kernel → Restart & Run All. Python 3 with NumPy and pandas, no internet, no API key. The simulation in B3 takes a few minutes.

**Data:** Kenneth R. French Data Library, value-weighted monthly returns of 30 U.S. industry portfolios and monthly factors (course snapshot, July 1926 – September 2026), in percent, divided by 100 on loading.

In [7]:
import os
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_var, "1")   # small 30x30 problems run faster single-threaded

from pathlib import Path
from functools import partial
import time
import numpy as np
import pandas as pd
from IPython.display import display
from QAM26_shrinkage import shrinkage
from QAM26_optimization import mean_variance_long_only

GAMMA = 4
COST = 0.001                      # 10 basis points per unit of wealth traded
START, END = "1980-01", "2026-09"


def load_monthly(filename, columns=None):
    # Read a data-pack CSV, index by month, convert percent to decimals.
    for directory in (Path.cwd() / "QAM26_data", Path.cwd()):
        path = directory / filename
        if path.is_file():
            frame = pd.read_csv(path, index_col="month")
            frame.index = pd.PeriodIndex(frame.index, freq="M")
            if columns is not None:
                frame = frame[columns]
            return frame.astype(float) / 100
    raise FileNotFoundError(f"{filename} not found in QAM26_data/ or beside the notebook")


industries = load_monthly("industries30_returns.csv")          # total returns R_{i,t}
factors = load_monthly("factors.csv", ["RF", "Mkt-RF"])
rf = factors["RF"]
assert industries.shape == (1203, 30) and industries.index.equals(rf.index)
excess = industries.sub(rf, axis=0)                              # r_{i,t} = R_{i,t} - R_{f,t}
print(f"Data: {industries.index[0]} to {industries.index[-1]}, {len(industries)} months, {industries.shape[1]} industries")

Data: 1926-07 to 2026-09, 1203 months, 30 industries


## Part A — The engine and six rules

### A1. The engine
Row $t$ of the weights table holds $\theta_{t-1}$, chosen from the excess returns of months $t-120,\dots,t-1$ (with `shift=1`, Part B1: $t-119,\dots,t$). The strategy excess return is $r_{p,t}=\theta_{t-1}'r_t$; the remainder earns $R_f$. Turnover compares the drifted weights $\theta^+_{i,t-1}=\theta_{i,t-1}(1+R_{i,t})/(1+R_{f,t}+r_{p,t})$ with the next row $\theta_t$ (560 month-ends, the first purchase is not counted). The trade at the end of month $t$ is charged to month $t+1$. `gamma` only enters the certainty equivalent (A6 d).

In [8]:
def performance(series, gamma=GAMMA):
    # Annualised measures (decimals) of monthly excess returns.
    mean = 12 * series.mean()
    vol = np.sqrt(12) * series.std(ddof=1)
    return {"Mean (%)": 100 * mean, "Volatility (%)": 100 * vol,
            "Sharpe": mean / vol if vol > 0 else np.nan,
            "CE (%)": 100 * (mean - gamma * vol**2 / 2)}


def run_backtest(returns, rf, rule, window=120, start="1980-01", end="2026-09",
                 cost=0.0, shift=0, gamma=GAMMA):
    # Rolling out-of-sample backtest. returns: total returns (decimals); rf: risk-free rate (decimals).
    returns, rf = returns.copy(deep=False), rf.copy(deep=False)
    for obj in (returns, rf):
        obj.index = pd.PeriodIndex(obj.index, freq="M")
    months = returns.index
    if (not months.equals(rf.index)
            or not months.equals(pd.period_range(months[0], months[-1], freq="M"))):
        raise ValueError("Industry and RF months must match, be ordered and have no gaps.")
    if window < 2 or cost < 0 or shift not in (0, 1) or gamma <= 0:
        raise ValueError("Invalid window, cost, shift or risk aversion.")
    first = months.get_loc(pd.Period(start, "M"))
    last = months.get_loc(pd.Period(end, "M"))
    if first < window:
        raise ValueError(f"returns has only {first} months before {start}, but the window needs {window}.")
    if last <= first:
        raise ValueError("At least two out-of-sample months are needed.")
    x = returns.sub(rf, axis=0)
    if not np.isfinite(x.iloc[first - window:last + 1 + shift].to_numpy()).all():
        raise ValueError("Missing or non-finite return in the analysis period.")

    held = np.empty((last - first + 1, returns.shape[1]))
    for row, t in enumerate(range(first, last + 1)):
        stop = t + shift                                   # slice end is exclusive
        weights = np.asarray(rule(x.iloc[stop - window:stop]), dtype=float).reshape(-1)
        if weights.shape != (held.shape[1],) or not np.isfinite(weights).all():
            raise ValueError(f"Invalid weights for {months[t]}")
        held[row] = weights

    period = slice(first, last + 1)
    gross = np.einsum("ti,ti->t", held, x.iloc[period].to_numpy())          # r_{p,t} before costs
    growth = 1 + rf.iloc[period].to_numpy() + gross                         # wealth growth
    drift = held[:-1] * (1 + returns.iloc[first:last].to_numpy()) / growth[:-1, None]
    turnover = np.abs(held[1:] - drift).sum(axis=1)                          # TO_t, 560 values
    net = pd.Series(gross - cost * np.r_[0, turnover], index=months[period], name="Excess return")

    measures = performance(net, gamma)
    measures["Turnover"] = turnover.mean()
    measures["Gross exposure"] = np.abs(held).sum(axis=1).mean()
    return {"returns": net,
            "weights": pd.DataFrame(held, index=net.index, columns=returns.columns),
            "measures": measures}

### A2. Six rules
Each rule receives only the estimation window (excess returns) and returns 30 weights. `np.linalg.solve(S, m)` computes $S^{-1}m$ without forming the inverse. Rules 3 and 4 leave the rest in (or borrow at) the risk-free rate; rules 1, 2, 5 and 6 sum to one; only rules 3, 4 and 6 can short.

In [9]:
def moments(data):
    x = np.asarray(data, dtype=float)
    return x.mean(axis=0), np.cov(x, rowvar=False, ddof=1)


def rule_equal_weights(window_data):                 # 1. 1/N
    N = window_data.shape[1]
    return np.ones(N) / N


def rule_inverse_volatility(window_data):            # 2. 1/sigma_i, scaled to sum to one
    inverse_sd = 1 / np.asarray(window_data, dtype=float).std(axis=0, ddof=1)
    return inverse_sd / inverse_sd.sum()


def rule_plugin(window_data, gamma=GAMMA):           # 3. (1/gamma) Sigma^-1 mu, sample moments
    mu, Sigma = moments(window_data)
    return np.linalg.solve(Sigma, mu) / gamma


def rule_shrinkage(window_data, gamma=GAMMA):        # 4. supplied James-Stein / Ledoit-Wolf estimators
    mu, Sigma, delta = shrinkage(np.asarray(window_data, dtype=float), 1)
    return np.linalg.solve(Sigma, mu) / gamma


def rule_long_only(window_data):                     # 5. theta >= 0, sum = 1, supplied optimiser
    mu, Sigma = moments(window_data)
    return mean_variance_long_only(mu, Sigma, GAMMA)


def rule_minimum_variance(window_data):              # 6. Sigma^-1 1 / (1' Sigma^-1 1)
    _, Sigma = moments(window_data)
    raw = np.linalg.solve(Sigma, np.ones(Sigma.shape[0]))
    return raw / raw.sum()


rules = {"1. 1/N": rule_equal_weights, "2. Inverse volatility": rule_inverse_volatility,
         "3. Plug-in MV": rule_plugin, "4. Shrinkage MV": rule_shrinkage,
         "5. Long-only MV": rule_long_only, "6. Minimum variance": rule_minimum_variance}
names = list(rules)
gross = {n: run_backtest(industries, rf, rule) for n, rule in rules.items()}
net = {n: run_backtest(industries, rf, rule, cost=COST) for n, rule in rules.items()}

### A3. Check values
The engine must reproduce the reference values for rules 1 and 2 (1980/01–2026/09, before costs).

In [10]:
summary = pd.DataFrame({n: r["measures"] for n, r in gross.items()}).T
checks = summary.iloc[:2][["Mean (%)", "Volatility (%)", "Sharpe", "Turnover"]].round(
    {"Mean (%)": 2, "Volatility (%)": 2, "Sharpe": 2, "Turnover": 3})
np.testing.assert_allclose(checks, [[9.04, 16.27, 0.56, 0.031], [8.89, 15.50, 0.57, 0.029]])
display(checks)

,Mean (%),Volatility (%),Sharpe,Turnover
1. 1/N,9.04,16.27,0.56,0.031
2. Inverse volatility,8.89,15.50,0.57,0.029


**Checks of the engine and of rules 3 to 6** (no reference values are given, so we test the defining conditions):
the handout's two-asset drift example, the error for a too-short history, the absence of look-ahead, and the first-order conditions of each optimisation.

In [11]:
# 1. Drift example of the handout: (0.5, 0.5), returns 10% and 0%, Rf = 0 -> trade 1/21 = 0.048; cost hits the next month.
toy_months = pd.period_range("2000-01", periods=4, freq="M")
toy = pd.DataFrame({"A": [0., 0., .10, 0.], "B": [0., 0., 0., 0.]}, index=toy_months)
toy_rf = pd.Series(0., index=toy_months)
demo = run_backtest(toy, toy_rf, rule_equal_weights, window=2, start="2000-03", end="2000-04", cost=COST)
np.testing.assert_allclose(demo["measures"]["Turnover"], 1 / 21)
np.testing.assert_allclose(demo["returns"], [.05, -COST / 21])

# 2. Too little history must stop with an error.
try:
    run_backtest(toy, toy_rf, rule_equal_weights, window=3, start="2000-03", end="2000-04")
except ValueError as error:
    assert "window needs" in str(error)
else:
    raise AssertionError("Insufficient history was accepted.")

# 3. No look-ahead: the first window is 1970/01-1979/12, and changing the return of a month
#    does not change the weights held in that month.
seen = []
run_backtest(industries, rf, lambda w: seen.append((w.index[0], w.index[-1], len(w))) or rule_equal_weights(w), end="1980-02")
assert seen[0] == (pd.Period("1970-01", "M"), pd.Period("1979-12", "M"), 120)
changed = industries.copy()
changed.loc["1980-06"] += 0.05
np.testing.assert_allclose(run_backtest(changed, rf, rule_plugin, end="1980-06")["weights"],
                           run_backtest(industries, rf, rule_plugin, end="1980-06")["weights"])

# 4. Structure of every result; costs equal c x turnover, charged one month later.
for i, n in enumerate(names):
    w = gross[n]["weights"]
    assert set(gross[n]) == {"returns", "weights", "measures"} and w.shape == (561, 30)
    charges = gross[n]["returns"] - net[n]["returns"]
    assert charges.iloc[0] == 0
    np.testing.assert_allclose(charges.iloc[1:].mean(), COST * summary.loc[n, "Turnover"])
    if i in (0, 1, 4, 5):
        np.testing.assert_allclose(w.sum(axis=1), 1)        # fully invested
    if i in (0, 1, 4):
        assert w.min().min() >= -1e-12                       # no short positions

# 5. First-order conditions of rules 3-6 in three windows.
for month in ["1980-01", "2008-10", "2026-09"]:
    pos = excess.index.get_loc(pd.Period(month, "M"))
    x = excess.iloc[pos - 120:pos]
    mu, Sigma = moments(x)
    w3, w4, w5, w6 = (gross[n]["weights"].loc[month].to_numpy() for n in names[2:])
    np.testing.assert_allclose(GAMMA * Sigma @ w3, mu, atol=1e-12)                 # rule 3
    mu_s, Sigma_s, _ = shrinkage(x.to_numpy(), 1)
    np.testing.assert_allclose(GAMMA * Sigma_s @ w4, mu_s, atol=1e-12)             # rule 4
    grad = mu - GAMMA * Sigma @ w5                                                  # rule 5 (KKT)
    active = w5 > 1e-10
    np.testing.assert_allclose(grad[active], grad[active].mean(), atol=1e-8)
    assert np.all(grad[~active] <= grad[active].mean() + 1e-8)
    np.testing.assert_allclose(Sigma @ w6, np.full(30, w6 @ Sigma @ w6), atol=1e-12)  # rule 6
print("All engine and rule checks passed.")

All engine and rule checks passed.


### A4. The table
Mean, volatility and CE in percent per year; Sharpe ratio, turnover and gross exposure as plain numbers. The last three columns are net of costs of 10 basis points. The market row is Mkt-RF over the same 561 months.

In [12]:
table = summary.copy()
for col in ["Mean (%)", "Sharpe", "CE (%)"]:
    table["Net " + col] = [net[n]["measures"][col] for n in names]
table.loc["Market (Mkt-RF)"] = pd.Series(performance(factors.loc[START:END, "Mkt-RF"]))
digits = {c: 2 for c in table.columns}
digits.update({"Sharpe": 3, "Net Sharpe": 3, "Turnover": 3})
# Formatted copy for display only (plain pandas, no jinja2 needed); `table` stays numeric.
shown = table.apply(lambda col: col.map(lambda v: "—" if pd.isna(v) else f"{v:.{digits[col.name]}f}"))
display(shown)

,Mean (%),Volatility (%),Sharpe,CE (%),Turnover,Gross exposure,Net Mean (%),Net Sharpe,Net CE (%)
1. 1/N,9.04,16.27,0.555,3.74,0.031,1.00,9.00,0.553,3.70
2. Inverse volatility,8.89,15.50,0.573,4.08,0.029,1.00,8.85,0.571,4.05
3. Plug-in MV,28.93,86.00,0.336,-119.00,9.189,30.45,17.92,0.208,-130.57
4. Shrinkage MV,20.84,34.69,0.601,-3.23,1.129,6.53,19.48,0.562,-4.54
5. Long-only MV,10.41,19.80,0.526,2.56,0.198,1.00,10.17,0.514,2.33
6. Minimum variance,7.59,13.02,0.583,4.20,0.471,4.12,7.03,0.540,3.64
Market (Mkt-RF),8.86,15.55,0.570,4.02,—,—,—,—,—


### A5. The in-sample Sharpe ratio
$\hat\mu$ and $\hat\Sigma$ of the 30 industry excess returns over the 561 out-of-sample months, and $\sqrt{12\,\hat\mu'\hat\Sigma^{-1}\hat\mu}$.

In [13]:
def tangency_sharpe(mu, Sigma):
    return np.sqrt(12 * mu @ np.linalg.solve(Sigma, mu))

sr_in_sample = tangency_sharpe(*moments(excess.loc[START:END]))
print(f"In-sample Sharpe ratio, 1980/01-2026/09:  {sr_in_sample:.3f}")
print(f"Out-of-sample Sharpe ratio of rule 3:    {table.loc[names[2], 'Sharpe']:.3f} "
      f"(net of costs {table.loc[names[2], 'Net Sharpe']:.3f})")

In-sample Sharpe ratio, 1980/01-2026/09:  1.072
Out-of-sample Sharpe ratio of rule 3:    0.336 (net of costs 0.208)


### A6. Questions

**(a)** The in-sample Sharpe ratio is **1.072**; the out-of-sample Sharpe ratio of rule 3 is **0.336** (0.208 net of costs). The in-sample number chooses the weights $\hat\Sigma^{-1}\hat\mu$ with the moments of the same 561 months on which it is evaluated. It is the maximum *ex-post* Sharpe ratio: every bit of estimation noise in $\hat\mu$ and $\hat\Sigma$ that happened to pay off is counted as skill, so it is biased upward. Rule 3 only knows the previous 120 months; the noise in $\hat\mu$, amplified by $\hat\Sigma^{-1}$, produces extreme positions (gross exposure 30.45) that do not pay off in the following month. Only the **0.336** could have been earned: the 1.072 requires knowing in January 1980 the returns of 1980–2026.

In [14]:
# A6(b): standard errors, and a paired delta-method check that accounts for the correlation between rules
# (months treated as independent; influence function of the annualised Sharpe ratio).
T = len(gross[names[0]]["returns"])
se_one, se_diff = np.sqrt(12 / T), np.sqrt(24 / T)
rp = pd.DataFrame({n: gross[n]["returns"] for n in names})
sd = rp.std(ddof=0)
zscore = (rp - rp.mean()) / sd
influence = np.sqrt(12) * (zscore - 0.5 * (rp.mean() / sd) * (zscore**2 - 1))
pairs = []
for i, a in enumerate(names):
    for b in names[i + 1:]:
        se = (influence[a] - influence[b]).std(ddof=1) / np.sqrt(T)
        diff = table.loc[a, "Sharpe"] - table.loc[b, "Sharpe"]
        pairs.append({"Pair": f"{a[:2]} vs {b[:2]}", "SR difference": diff, "Correlation": rp[a].corr(rp[b]),
                      "Paired SE": se, "z": diff / se})
pairs = pd.DataFrame(pairs).set_index("Pair").sort_values("z", key=abs, ascending=False)
print(f"SE of one Sharpe ratio: {se_one:.3f};  SE of a difference (uncorrelated): {se_diff:.3f}")
display(pairs.head(6).round(3))

SE of one Sharpe ratio: 0.146;  SE of a difference (uncorrelated): 0.207


,SR difference,Correlation,Paired SE,z
Pair,,,,
3. vs 4.,-0.264,0.555,0.141,-1.875
1. vs 2.,-0.018,0.997,0.012,-1.539
3. vs 6.,-0.247,0.340,0.169,-1.458
2. vs 3.,0.237,0.197,0.191,1.243
3. vs 5.,-0.189,0.460,0.156,-1.216
1. vs 3.,0.219,0.179,0.192,1.140


**(b)** With $T=561$ the standard error of one Sharpe ratio is $\sqrt{12/561}=$ **0.146**, that of a difference **0.207** if the returns are uncorrelated. Rules 1, 2, 4, 5 and 6 lie between 0.526 and 0.601: the widest gap, 0.075, is about a third of a standard error, so **these five cannot be ranked**. Positive correlation shrinks the standard error of a difference (1/N and inverse volatility have correlation 1.00, so even their gap of 0.018 has a paired SE of only 0.012), but the paired check above finds no difference beyond $|z|=1.96$; the largest is 1.87 (rule 3 vs rule 4). Rule 3 (0.336) lies 0.19–0.26 below the others, 0.9–1.3 standard errors of an uncorrelated difference: weak evidence that it is worse, not a reliable ranking.

What sets rule 3 apart are the other columns: volatility **86.00%** a year (13–35% for the others), certainty equivalent **−119.00%**, turnover **9.19** (it trades about nine times its wealth every month) and gross exposure **30.45** (long and short positions of thirty times wealth). Plug-in mean-variance turns estimation error into huge, unstable, leveraged bets: a $\gamma=4$ investor would rather hold T-bills (CE = 0), and after costs its Sharpe ratio falls to 0.208.

In [15]:
ranked = pd.DataFrame({c: table.loc[names, c].sort_values(ascending=False).index for c in ["Sharpe", "Net Sharpe"]},
                      index=pd.RangeIndex(1, 7, name="Rank"))
ranked["Turnover of the rule (net order)"] = table.loc[ranked["Net Sharpe"], "Turnover"].round(3).to_numpy()
display(ranked)

,Sharpe,Net Sharpe,Turnover of the rule (net order)
Rank,,,
1,4. Shrinkage MV,2. Inverse volatility,0.029
2,6. Minimum variance,4. Shrinkage MV,1.129
3,2. Inverse volatility,1. 1/N,0.031
4,1. 1/N,6. Minimum variance,0.471
5,5. Long-only MV,5. Long-only MV,0.198
6,3. Plug-in MV,3. Plug-in MV,9.189


**(c)** Before costs: 4 (0.601) > 6 (0.583) > 2 (0.573) > 1 (0.555) > 5 (0.526) > 3 (0.336).
Net of costs: 2 (0.571) > 4 (0.562) > 1 (0.553) > 6 (0.540) > 5 (0.514) > 3 (0.208).
Inverse volatility moves from 3rd to 1st and 1/N from 4th to 3rd; shrinkage drops from 1st to 2nd and minimum variance from 2nd to 4th; rules 5 and 3 keep their places. The **turnover** column explains it: the annual cost drag is about $12\,c\,TO$, i.e. 0.04% for rules 1 and 2 (turnover 0.031 and 0.029), 0.56% for rule 6 (0.471), 1.35% for rule 4 (1.129) and 11.0% for rule 3 (9.189). These rank changes are much smaller than the standard errors in (b).

**(d) First predict, then run.**
*Prediction (written before running).* Rule 3 holds $\frac1\gamma\hat\Sigma^{-1}\hat\mu$: changing $\gamma$ only scales every industry weight by $4/\gamma$ and puts the rest in T-bills. The excess return scales by $4/\gamma$, so mean and volatility scale by the same factor and the **Sharpe ratio does not change**. With $m_4$ and $s_4$ the values at $\gamma=4$:
$$CE(\gamma)=\tfrac4\gamma m_4-\tfrac\gamma2\left(\tfrac4\gamma\right)^2 s_4^2=\tfrac4\gamma\,(m_4-2s_4^2)=\tfrac4\gamma\,CE(4).$$
So the CE **moves toward zero but keeps its sign**: about −119% × 4/6 ≈ −79% for $\gamma=6$ and −119% × 4/8 ≈ −60% for $\gamma=8$.

In [16]:
sensitivity = {}
for g in (4, 6, 8):
    result = run_backtest(industries, rf, partial(rule_plugin, gamma=g), gamma=g)
    sensitivity[f"gamma = {g}"] = {k: result["measures"][k] for k in ("Mean (%)", "Volatility (%)", "Sharpe", "CE (%)")}
    np.testing.assert_allclose(result["returns"], gross[names[2]]["returns"] * 4 / g, atol=1e-14)
sensitivity = pd.DataFrame(sensitivity).T
display(sensitivity.round(3))

,Mean (%),Volatility (%),Sharpe,CE (%)
gamma = 4,28.928,86.002,0.336,-119.000
gamma = 6,19.285,57.335,0.336,-79.334
gamma = 8,14.464,43.001,0.336,-59.500


*Result.* As predicted: the Sharpe ratio is 0.336 for all three values of $\gamma$, and the CE is −119.00%, −79.33% and −59.50%.

**No value of $\gamma$ makes the CE positive.** Let $a$ and $b$ be the annual mean and volatility of the unscaled portfolio $\hat\Sigma^{-1}\hat\mu$. For any $\gamma$, $m=a/\gamma$, $s=b/\gamma$ and $CE=(a-b^2/2)/\gamma$: $\gamma$ changes the size of the CE, never its sign. The sign is set by the direction of the portfolio, and here $a-b^2/2=4\,CE(4)<0$. Equivalently, $CE>0$ requires $SR>\gamma s/2=b/2=2s_4=1.72$, while the realised Sharpe ratio is 0.336. As $\gamma\to\infty$ the investor holds only T-bills and the CE approaches 0 from below. The problem is the direction of the bet (estimation error), not its scale.

## Part B — Three experiments

### B1. One month of look-ahead
With `shift=1` the window is $t-119,\dots,t$: it contains the return of the month in which the portfolio is held.

In [17]:
b1 = pd.DataFrame({n: {"shift = 0": table.loc[n, "Sharpe"],
                       "shift = 1": run_backtest(industries, rf, rules[n], shift=1)["measures"]["Sharpe"]}
                   for n in (names[0], names[2], names[3])}).T
b1["Difference"] = b1["shift = 1"] - b1["shift = 0"]
display(b1.round(3))

,shift = 0,shift = 1,Difference
1. 1/N,0.555,0.555,0.000
3. Plug-in MV,0.336,2.337,2.000
4. Shrinkage MV,0.601,1.245,0.644


1/N is unchanged (0.555): its weights do not depend on the data. Rule 4 rises from 0.601 to 1.245 and **rule 3 from 0.336 to 2.337**. With `shift=1`, $\hat\mu$ contains $r_t/120$, so the weights contain the term $\frac{1}{120\gamma}\hat\Sigma^{-1}r_t$. Its contribution to the return of month $t$ is $\frac{1}{120\gamma}\,r_t'\hat\Sigma^{-1}r_t$, which is **positive in every month** (a quadratic form). $\hat\Sigma^{-1}$ is large in the directions in which the sample shows little risk, so this small look-ahead is blown up by the same mechanism that gives rule 3 a gross exposure of 30. Rule 4 shrinks the mean toward the grand mean and the covariance matrix toward a well-conditioned target, which damps both effects, so it gains less. The more a rule relies on $\hat\mu$ and $\hat\Sigma^{-1}$, the more a one-month timing error inflates its backtest.

### B2. The length of the window, in the data
The out-of-sample months stay 1980/01–2026/09; a 240-month window starts in 1960/01.

In [18]:
windows = (60, 120, 240)
b2_results = {(n, w): run_backtest(industries, rf, rules[n], window=w) for n in names[2:4] for w in windows}
b2 = pd.DataFrame({n: [b2_results[(n, w)]["measures"]["Sharpe"] for w in windows] for n in names[2:4]},
                  index=pd.Index(windows, name="Window (months)"))
display(b2.round(3))
worst_month = (b2_results[(names[2], 60)]["returns"] + rf.loc[START:END]).min()
print(f"Rule 3, 60-month window: worst monthly total return {worst_month:.1%}")

,3. Plug-in MV,4. Shrinkage MV
Window (months),,
60,0.265,0.477
120,0.336,0.601
240,0.179,0.505


Rule 3, 60-month window: worst monthly total return -492.1%


Both rules do best with 120 months. Note that rule 3 with a 60-month window has a month with a total return of about −492%: with so few observations the positions become so extreme that the investor would have lost more than their entire wealth. The Sharpe ratio is computed from the return series as required, but such a strategy could not have been continued without fresh capital.

### B3. The length of the window, in a world that does not change
**1. The world.** $\mu$ and $\Sigma$ are the sample moments of the 30 industry excess returns over 1960/01–2026/09 (801 months).

In [19]:
calibration = excess.loc["1960-01":END]
assert len(calibration) == 801
mu_true, Sigma_true = moments(calibration)
theta_star = np.linalg.solve(Sigma_true, mu_true) / GAMMA
sr_star = tangency_sharpe(mu_true, Sigma_true)
print(f"True Sharpe ratio SR* = {sr_star:.3f}")


def rule_theta_star(window_data):
    # Knows the true parameters: always holds theta*.
    return theta_star

True Sharpe ratio SR* = 0.890


**2.–4. 100 histories.** Each history has 801 i.i.d. normal months labelled 1960/01–2026/09 with $R_f=0$, so total and excess returns coincide. On each history the same engine runs rule 1, $\theta^*$, and rules 3 and 4 with windows of 60, 120 and 240 months (out-of-sample 1980/01–2026/09). The seed is set once, before the first history; all eight cases of a history use the same draws, so window comparisons are paired.

In [ ]:
SEED = 20261123
N_HISTORIES = 100
rng = np.random.default_rng(SEED)
sim_months = pd.period_range("1960-01", END, freq="M")
zero_rf = pd.Series(0.0, index=sim_months)
cases = [("1. 1/N", rule_equal_weights, 120), ("theta*", rule_theta_star, 120)] + \
        [(f"3. Plug-in MV, {w}", rule_plugin, w) for w in windows] + \
        [(f"4. Shrinkage MV, {w}", rule_shrinkage, w) for w in windows]

started = time.perf_counter()
sim = []
for h in range(N_HISTORIES):
    history = rng.multivariate_normal(mu_true, Sigma_true, size=801)
    history = pd.DataFrame(history, index=sim_months, columns=industries.columns)
    sim.append({label: run_backtest(history, zero_rf, rule, window=w)["measures"]["Sharpe"]
                for label, rule, w in cases})
sim = pd.DataFrame(sim)
assert sim.shape == (100, 8) and np.isfinite(sim.to_numpy()).all()
print(f"Seed {SEED}, {N_HISTORIES} histories, {time.perf_counter() - started:.0f} s. True SR* = {sr_star:.3f}")
sim_summary = pd.DataFrame({"Average Sharpe ratio": sim.mean(), "SD across histories": sim.std(ddof=1)})
display(sim_summary.round(3))

In [ ]:
# B3(b): direction and size of the 240- vs 120-month comparison for rule 3.
change = sim["3. Plug-in MV, 240"] - sim["3. Plug-in MV, 120"]
drop_data = b2.loc[240, names[2]] - b2.loc[120, names[2]]
lower = int((change < 0).sum())
as_large = int((change <= drop_data).sum())
print(f"Rule 3: SR(240) < SR(120) in {lower} of {N_HISTORIES} histories")
print(f"Average change 240 - 120 in the simulation: {change.mean():+.3f};  largest drop: {change.min():+.3f}")
print(f"Change in the data (B2): {drop_data:+.3f};  histories with a drop at least as large: {as_large}")

# B3(c): how much cross-sectional dispersion of the sample means would sampling noise alone create?
M = np.eye(30) - 1 / 30                                        # demeaning across industries
noise_var = np.trace(M @ Sigma_true @ M) / 29 / len(calibration)
print(f"Cross-sectional SD of the 30 sample means: {1200 * mu_true.std(ddof=1):.2f}% a year")
print(f"Expected SD created by estimation noise alone: {1200 * np.sqrt(noise_var):.2f}% a year")

**(a)** In the simulated world both rules **improve steadily with the window**: rule 3 0.186 / 0.317 / 0.472 and rule 4 0.315 / 0.414 / 0.535 for 60 / 120 / 240 months, all far below $\theta^*$ (0.885; SR* = 0.890). The parameters never change, so old months are as informative as new ones and a longer window only reduces estimation error. Rule 3 gains most because it suffers most from estimation error; shrinkage already removes part of it and gains less. **In the data (B2) the pattern is hump-shaped:** rule 3 0.265 / 0.336 / 0.179, rule 4 0.477 / 0.601 / 0.505. From 60 to 120 months the reduction of estimation error dominates; from 120 to 240 months the older data describe a different world (means and covariances change over time), and the bias they bring outweighs the further reduction of noise. The difference between the two patterns is this non-stationarity, which the simulation removes by construction, plus the luck of observing only one history (b).

**(b)** In **9 of 100** histories rule 3 has a lower Sharpe ratio with 240 months than with 120. So the *direction* seen in B2 occurs by chance in about one stationary history in eleven. Its *size* does not: on average the longer window gains +0.155, the largest drop in the 100 histories is −0.118, and a drop as large as in the data (−0.158) occurs in **none** of them. The result of B2 is therefore hard to explain by sampling noise alone and is evidence that the parameters change over time. It is not proof: the data are one history, and real returns are not i.i.d. normal (fat tails, volatility clustering), which the simulation ignores.

**(c)** **By less.** Each sample mean is the true mean plus estimation error, so in expectation the dispersion of the 30 sample means is the dispersion of the true means plus the dispersion of the errors. Here, estimation noise alone would create a cross-sectional standard deviation of about 1.77% a year, more than the 1.56% that $\mu$ shows: the observed differences are fully compatible with noise. The simulated world therefore builds in larger differences in expected returns than the real world probably has, which gives rules 3 and 4 something to learn that may not exist. Outside the simulation 1/N, which estimates nothing, should therefore look *better* relative to rules 3 and 4. Even inside the simulation, rules 3 and 4 with a 120-month window fall short of 1/N (0.317 and 0.414 vs 0.460). In the data rule 3 is far below 1/N (0.336 vs 0.555), and rule 4's lead (0.601 vs 0.555) is only a fraction of a standard error.

### Consistency check
Every number quoted in the written answers is compared with the computed results, so text and tables cannot drift apart after a re-run.

In [ ]:
def cited(value, quoted, digits=3):
    assert round(float(value), digits) == quoted, f"text says {quoted}, computed {value:.6f}"

S, N3 = table["Sharpe"], names[2]
for n, q in zip(names, (0.555, 0.573, 0.336, 0.601, 0.526, 0.583)):
    cited(S[n], q)
for n, q in zip(names, (0.553, 0.571, 0.208, 0.562, 0.514, 0.540)):
    cited(table.loc[n, "Net Sharpe"], q)
cited(sr_in_sample, 1.072); cited(se_one, 0.146); cited(se_diff, 0.207)
cited(table.loc[N3, "Volatility (%)"], 86.00, 2); cited(table.loc[N3, "CE (%)"], -119.00, 2)
cited(table.loc[N3, "Turnover"], 9.19, 2); cited(table.loc[N3, "Gross exposure"], 30.45, 2)
cited(abs(pairs["z"]).max(), 1.87, 2); cited(pairs.loc["1. vs 2.", "Paired SE"], 0.012)
cited(sensitivity["CE (%)"].iloc[1], -79.33, 2); cited(sensitivity["CE (%)"].iloc[2], -59.50, 2)
cited(b1.iloc[1, 1], 2.337); cited(b1.iloc[2, 1], 1.245)
for n, qs in zip(names[2:4], ((0.265, 0.336, 0.179), (0.477, 0.601, 0.505))):
    for w, q in zip(windows, qs):
        cited(b2.loc[w, n], q)
cited(worst_month * 100, -492, 0)
cited(sr_star, 0.890)
for label, q in {"1. 1/N": 0.460, "theta*": 0.885, "3. Plug-in MV, 60": 0.186, "3. Plug-in MV, 120": 0.317,
                 "3. Plug-in MV, 240": 0.472, "4. Shrinkage MV, 60": 0.315, "4. Shrinkage MV, 120": 0.414,
                 "4. Shrinkage MV, 240": 0.535}.items():
    cited(sim[label].mean(), q)
assert lower == 9 and as_large == 0
cited(change.mean(), 0.155); cited(change.min(), -0.118); cited(drop_data, -0.158)
cited(1200 * mu_true.std(ddof=1), 1.56, 2); cited(1200 * np.sqrt(noise_var), 1.77, 2)
print("All numbers quoted in the text match the computed results.")